In [1]:
import sys
sys.path.append("/Users/treveralexander/Library/CloudStorage/OneDrive-EY/Personal/Football-Stats/Fantasy_Football")

In [2]:
import os
import itertools
import time
from typing import Union, Iterable
import polars as pl
import psutil
from Logging.AuditLogger import AuditLogger
import Utility.utility as util
import Helper.helper as help

def extract_and_save_player_stats(
    categories: Union[str, Iterable[str]],
    years: Union[int, Iterable[int]],
    base_path: str,
    source_system: str,
    destination_system: str,
    audit_logger: AuditLogger
) -> None:
    """
    Extracts player statistics for specified categories and years, creates DataFrames,
    and saves them as JSON files in a structured directory.

    Args:
        categories: Player statistic categories (e.g., "passing", "rushing", "receiving").
        years: Years to extract data for (1970-current year).
        base_path: Base path for saving the JSON files.
        source_system: Name of the system where the original data is being extracted from.
        destination_system: Name of the system where the processed data will be stored.
        audit_logger: Instance of AuditLogger for logging.
    """
    current_season = util.current_nfl_regular_season()
    
    categories = util.ensure_iterable(categories)
    years = util.ensure_iterable(years)

    valid_categories = util.get_valid_player_categories(categories)

    for category, year in itertools.product(categories, years):
        pipeline_name = f"extract_and_save_player_stats({category})"
        start_time = time.time()
        try:
            _validate_inputs(category, year, valid_categories, current_season)
            
            category_path = os.path.join(base_path, category.lower())
            os.makedirs(category_path, exist_ok=True)

            data, headers = help.get_stats(category, year)
            df = pl.DataFrame(data, schema=headers)
            
            file_path = os.path.join(category_path, f"{category}_{year}.json")
            util.write_json(file_path, df)

            _log_success(audit_logger, pipeline_name, start_time, source_system, destination_system, len(df), category, year)
        except Exception as e:
            _log_failure(audit_logger, pipeline_name, start_time, source_system, destination_system, str(e))

    print("Pipeline finished executing!")

def _validate_inputs(category: str, year: int, valid_categories: set, current_season: int) -> None:
    if year < 1970 or year > current_season:
        raise ValueError(f"Invalid year provided: {year}")
    if category not in valid_categories:
        raise ValueError(f"Invalid category provided: {category}")

def _log_success(audit_logger: AuditLogger, pipeline_name: str, start_time: float, source_system: str, destination_system: str, 
                 num_records: int, category: str, year: int) -> None:
    end_time = time.time()
    processing_time = end_time - start_time
    audit_logger.log(
        pipeline_name=pipeline_name,
        start_time=time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)),
        end_time=time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)),
        source_system=source_system,
        destination_system=destination_system,
        num_records=num_records,
        processing_time=processing_time,
        cpu_usage=psutil.cpu_percent(),
        memory_usage=psutil.virtual_memory().percent,
        status="Success",
        status_message=f"Pipeline execution completed successfully for category: {category} and year: {year}"
    )

def _log_failure(audit_logger: AuditLogger, pipeline_name: str, start_time: float, source_system: str, destination_system: str, error_message: str) -> None:
    end_time = time.time()
    processing_time = end_time - start_time
    audit_logger.log(
        pipeline_name=pipeline_name,
        start_time=time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)),
        end_time=time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)),
        source_system=source_system,
        destination_system=destination_system,
        num_records=0,
        processing_time=processing_time,
        cpu_usage=psutil.cpu_percent(),
        memory_usage=psutil.virtual_memory().percent,
        status="Failure",
        status_message=f"Error executing pipeline: {error_message}"
    )

In [4]:
# Example usage
if __name__ == "__main__":
    log_file_path = "/Users/treveralexander/Library/CloudStorage/OneDrive-EY/Personal/Football-Stats/Storage/Logs/audit_logs.csv"
    audit_logger = AuditLogger(log_file_path)

    extract_and_save_player_stats(
        categories=["passing", "rushing"],
        years=[2023],
        base_path="/Users/treveralexander/Library/CloudStorage/OneDrive-EY/Personal/Football-Stats/Storage/Raw/Player_Stats",
        source_system="NFL API",
        destination_system="Local Storage",
        audit_logger=audit_logger
    )

Pipeline finished executing!
